In [ ]:
import os
import sys

# 让 notebook 能 import 同目录下的 BossDuelEnv.py。
# (原先是硬编码的 d:\VSCODEProject\PPO\enery,换台机器就失效。)
_HERE = os.path.dirname(os.path.abspath(__file__)) if "__file__" in globals() else os.getcwd()
if _HERE not in sys.path:
    sys.path.insert(0, _HERE)

# 避免 OpenMP 被重复加载时报错(常见于 torch + mujoco 共存的环境)
os.environ["KMP_DUPLICATE_LIB_OK"] = "True"

print("cwd:", os.getcwd())


In [ ]:
# 环境定义统一在 BossDuelEnv.py 里,这里直接导入。
#
# 注意:这个 cell 原本内联了一份旧的 BossDuelEnv 副本,参数是
#     boss_health_max = 150, boss_probs = [0.45, 0.5, 0.05], agent_ult_damage = 40
# 而训练用的 BossDuelEnv.py 是
#     boss_health_max = 200, boss_probs = [0.4, 0.5, 0.1],  agent_ult_damage = 30
#
# 两份不一致会让"评估"失去意义 —— 模型被放到一个它没训练过的环境里打分。
# 统一从 BossDuelEnv.py 导入,避免再次漂移。
from BossDuelEnv import BossDuelEnv

print(f"Boss HP 上限 = {BossDuelEnv().boss_health_max}"
      "   (应为 200,与训练一致)")


In [3]:
import numpy as np
from stable_baselines3 import PPO

def get_boss_action_name(state):
    """从状态中提取 Boss 当前动作名称"""
    boss_action = np.argmax(state[:3])
    return ["重击", "轻击", "蓄力"][boss_action]

def print_state(env, state, turn):
    """打印当前回合开始时的状态（Boss 即将使用的动作）"""
    print("\n" + "="*60)
    print(f"【第 {turn} 回合】")
    print(f"Agent HP: {env.agent_health:.1f} / {env.max_health}")
    print(f"Boss  HP: {env.boss_health:.1f} / {env.boss_health_max}")
    print(f"能量: {env.agent_energy:.1f} / {env.max_energy}")
    print(f"体力: {env.agent_stamina:.1f} / {env.max_stamina}")
    boss_action = get_boss_action_name(state)
    print(f"Boss 即将使用的动作: {boss_action}", end="")
    if boss_action == "蓄力":
        print("  ⚠️ 蓄力后下一回合攻击暴击！可用大招打断！")
    else:
        print()
    if env.boss_charge:
        print("🔥 Boss 当前处于蓄力状态！下回合攻击必定暴击且无法格挡/闪避！🔥")
    print("="*60)

def main():
    # 加载训练好的模型
    model_path = "ppo_treth.zip"   # 请根据实际文件名修改
    model = PPO.load(model_path)
    
    # 创建环境（gymnasium 风格）
    env = BossDuelEnv()
    obs, info = env.reset()   # 注意：gymnasium reset 返回 (obs, info)
    total_reward = 0
    turn = 1
    done = False

    print("AI 开始挑战 Boss！\n")

    while not done:
        # 显示当前状态
        print_state(env, obs, turn)
        
        # 模型预测动作（使用确定性策略）
        action, _ = model.predict(obs, deterministic=True)
        action_names = ["格挡", "闪避", "大招", "普攻"]
        print(f"AI 选择了: {action_names[action]}")
        
        # 执行动作（gymnasium step 返回 5 个值）
        obs, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated
        total_reward += reward
        
        # 显示本回合结果
        print("\n--- 本回合结果 ---")
        boss_prev = ["重击", "轻击", "蓄力"][info['boss_action']]
        agent_prev = action_names[action]
        print(f"Boss 实际动作: {boss_prev} | AI 动作: {agent_prev}")
        print(f"造成伤害: {info['damage_dealt']} | 受到伤害: {info['damage_taken']}")
        print(f"本回合奖励: {reward:.2f}")
        print(f"累计奖励: {total_reward:.2f}")
        
        if done:
            print("\n游戏结束！")
            if terminated:
                if env.agent_health <= 0:
                    print("AI 输了...")
                elif env.boss_health <= 0:
                    print("AI 赢了！")
            else:
                print("平局（超时）")
            print(f"最终累计奖励: {total_reward:.2f}")
            break
        
        turn += 1

if __name__ == "__main__":
    main()

c:\Users\MR\anaconda3\envs\Herta\Lib\site-packages\stable_baselines3\common\on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


AI 开始挑战 Boss！


【第 1 回合】
Agent HP: 100.0 / 100
Boss  HP: 150.0 / 150
能量: 0.0 / 100
体力: 100.0 / 100
Boss 即将使用的动作: 轻击
AI 选择了: 格挡

--- 本回合结果 ---
Boss 实际动作: 轻击 | AI 动作: 格挡
造成伤害: 0 | 受到伤害: 2
本回合奖励: -0.40
累计奖励: -0.40

【第 2 回合】
Agent HP: 98.0 / 100
Boss  HP: 150.0 / 150
能量: 15.0 / 100
体力: 100.0 / 100
Boss 即将使用的动作: 重击
AI 选择了: 闪避

--- 本回合结果 ---
Boss 实际动作: 重击 | AI 动作: 闪避
造成伤害: 0 | 受到伤害: 0
本回合奖励: 1.00
累计奖励: 0.60

【第 3 回合】
Agent HP: 98.0 / 100
Boss  HP: 150.0 / 150
能量: 20.0 / 100
体力: 85.0 / 100
Boss 即将使用的动作: 重击
AI 选择了: 格挡

--- 本回合结果 ---
Boss 实际动作: 重击 | AI 动作: 格挡
造成伤害: 0 | 受到伤害: 5
本回合奖励: -1.00
累计奖励: -0.40

【第 4 回合】
Agent HP: 93.0 / 100
Boss  HP: 150.0 / 150
能量: 35.0 / 100
体力: 90.0 / 100
Boss 即将使用的动作: 轻击
AI 选择了: 格挡

--- 本回合结果 ---
Boss 实际动作: 轻击 | AI 动作: 格挡
造成伤害: 0 | 受到伤害: 2
本回合奖励: -0.40
累计奖励: -0.80

【第 5 回合】
Agent HP: 91.0 / 100
Boss  HP: 150.0 / 150
能量: 50.0 / 100
体力: 95.0 / 100
Boss 即将使用的动作: 重击
AI 选择了: 闪避

--- 本回合结果 ---
Boss 实际动作: 重击 | AI 动作: 闪避
造成伤害: 0 | 受到伤害: 0
本回合奖励: 1.00
累计奖励: 0.20

【第 6 回合】
Ag